# 25.4 — Program synthesis

**Lesson tagline.** Program synthesis searches a language of possible programs until one matches the behavior we asked for.

Candidate programs are executable hypotheses. Examples score them, prune version spaces, and reveal ambiguity when the specification is too small.

Save a copy to Drive to edit.

In [ ]:

import math
import itertools
import random
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

SEED = 25
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(z):
    z = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))


def binary_cross_entropy(p, y):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1.0 - 1e-12)
    y = np.asarray(y, dtype=float)
    terms = -y * np.log(p) - (1.0 - y) * np.log(1.0 - p)
    return float(np.mean(terms))



@dataclass
class Program:
    name: str
    fn: object
    size: int


def enumerate_programs(constants=(-2, -1, 0, 1, 2), rich=False, branches=False):
    programs = []
    for c in constants:
        programs.append(Program(f'x+{c}', lambda x, c=c: x + c, 1))
        programs.append(Program(f'{c}*x', lambda x, c=c: c * x, 1))
        programs.append(Program(f'x^2+{c}', lambda x, c=c: x * x + c, 2))
    if rich:
        for a in constants:
            for b in constants:
                programs.append(Program(f'{a}*x+{b}', lambda x, a=a, b=b: a * x + b, 2))
                programs.append(Program(f'(x+{a})^2+{b}', lambda x, a=a, b=b: (x + a) * (x + a) + b, 3))
    if branches:
        for c in constants:
            for t in constants:
                programs.append(Program(f'if x>{t} then x^2+{c} else x+{c}', lambda x, c=c, t=t: np.where(x > t, x * x + c, x + c), 4))
    return programs


def score_program(program, examples):
    x = np.asarray([pair[0] for pair in examples], dtype=float)
    y = np.asarray([pair[1] for pair in examples], dtype=float)
    pred = np.asarray(program.fn(x), dtype=float)
    return float(np.mean((pred - y) ** 2))


def synthesize(programs, examples):
    scored = [(score_program(program, examples), program) for program in programs]
    return min(scored, key=lambda item: (item[0], item[1].size, item[1].name)), scored


def make_synthesis_ladder():
    rungs = []
    examples1 = [(0, 1), (-2, 5), (-1, 2), (1, 2), (2, 5)]
    rungs.append({'name': 'D1 lesson grammar', 'examples': examples1, 'heldout': [(3, 10), (4, 17)], 'programs': enumerate_programs()})
    examples2 = [(-3, 10), (-2, 5), (-1, 2), (0, 1), (1, 2), (2, 5), (3, 10)]
    rungs.append({'name': 'D2 more examples clean target', 'examples': examples2, 'heldout': [(4, 17)], 'programs': enumerate_programs()})
    rungs.append({'name': 'D3 ambiguous incomplete specs', 'examples': [(0, 1), (1, 2)], 'heldout': [(-2, 5), (2, 5)], 'programs': enumerate_programs()})
    rungs.append({'name': 'D4 richer composition grammar', 'examples': examples2, 'heldout': [(4, 17), (5, 26)], 'programs': enumerate_programs(rich=True)})
    x5 = np.array([-4, -3, -2, -1, 0, 1, 2, 3, 4])
    y5 = np.where(x5 > 0, x5 * x5 + 1, x5 + 1)
    y5 = y5.astype(float)
    y5[2] = y5[2] + 1.0
    examples5 = list(zip(x5.tolist(), y5.tolist()))
    rungs.append({'name': 'D5 branches noisy hidden target stress', 'examples': examples5, 'heldout': [(-5, -4), (5, 26)], 'programs': enumerate_programs(rich=True, branches=True)})
    return rungs


## The concept, built once (D1)

Synthesis minimizes $L(p)=rac1m\sum_i(p(x_i)-y_i)^2$ over a finite grammar $\mathcal{P}$. The lesson grammar contains $x+c$, $c\cdot x$, and $x^2+c$ for $c\in\{-2,-1,0,1,2\}$.

In [ ]:

programs = enumerate_programs()
examples = [(0, 1), (-2, 5), (-1, 2), (1, 2), (2, 5)]
lookup = {program.name: program for program in programs}
print('program count:', len(programs))
print('x^2+1 MSE:', score_program(lookup['x^2+1'], examples))
print('x^2+0 MSE:', score_program(lookup['x^2+0'], examples))
print('x^2+2 MSE:', score_program(lookup['x^2+2'], examples))
print('x+2 MSE:', score_program(lookup['x+2'], examples))
assert score_program(lookup['x^2+1'], examples) == 0.0
assert score_program(lookup['x^2+0'], examples) == 1.0
assert score_program(lookup['x^2+2'], examples) == 1.0
assert score_program(lookup['x+2'], examples) == 5.8


Version-space pruning counts exact survivors after each example. The plan's exact counts are $2,1,1,1,1$.

In [ ]:

counts = []
for m in range(1, len(examples) + 1):
    prefix = examples[:m]
    survivors = [program.name for program in programs if score_program(program, prefix) == 0.0]
    counts.append(len(survivors))
    print(m, survivors)
assert counts == [2, 1, 1, 1, 1]


## The dataset ladder

The F16 ladder expands examples, ambiguity, grammar richness, and finally branch/noise complexity.

In [ ]:

rungs = make_synthesis_ladder()
for i, rung in enumerate(rungs, start=1):
    print(f"D{i}: {rung['name']} examples={len(rung['examples'])} heldout={len(rung['heldout'])} programs={len(rung['programs'])}")
    print('sample examples:', rung['examples'][:4])


## Run the SAME method across D1–D5

In [ ]:

results = []
for i, rung in enumerate(rungs, start=1):
    best, scored = synthesize(rung['programs'], rung['examples'])
    heldout_mse = score_program(best[1], rung['heldout']) if rung['heldout'] else np.nan
    solved = int(best[0] == 0.0 and heldout_mse == 0.0)
    results.append({'D': i, 'best_name': best[1].name, 'mse': best[0], 'heldout': heldout_mse, 'searched': len(scored), 'solved': solved, 'scored': scored})
print('rung | best_program | train_mse | heldout_mse | solved | searched')
for row in results:
    print(f"D{row['D']} | {row['best_name']} | {row['mse']:.4f} | {row['heldout']:.4f} | {row['solved']} | {row['searched']}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, rung in enumerate(rungs):
    example_prefix_counts = []
    for m in range(1, len(rung['examples']) + 1):
        prefix = rung['examples'][:m]
        survivors = sum(score_program(program, prefix) == 0.0 for program in rung['programs'])
        example_prefix_counts.append(survivors)
    axes[0, col].plot(range(1, len(example_prefix_counts) + 1), example_prefix_counts, marker='o')
    axes[0, col].set_title(f"D{col + 1} survivors")
    axes[0, col].set_xlabel('examples')
    axes[0, col].set_ylabel('zero-loss programs')
axes[1, 0].plot([row['D'] for row in results], [row['mse'] for row in results], marker='o', label='best MSE')
axes[1, 0].plot([row['D'] for row in results], [row['searched'] for row in results], marker='s', label='programs searched')
axes[1, 0].set_title('MSE and search vs complexity')
axes[1, 0].legend()
for ax in axes[1, 1:]:
    ax.axis('off')
plt.tight_layout()
plt.show()


## Pitfall on the hardest rung

A weak spec can produce a wrong zero-training-loss program. Held-out tests and stronger examples expose the ambiguity.

In [ ]:

ambiguous = {'examples': [(0, 1)], 'heldout': [(-2, 5), (2, 5)], 'programs': enumerate_programs()}
best, scored = synthesize(ambiguous['programs'], ambiguous['examples'])
zero_train = [program.name for loss, program in scored if loss == 0.0]
print('zero-loss programs on one example:', zero_train)
for name in zero_train:
    print(name, 'heldout MSE', score_program(lookup[name], ambiguous['heldout']))
stronger = [(0, 1), (-2, 5), (2, 5)]
strong_best, strong_scored = synthesize(ambiguous['programs'], stronger)
print('after stronger examples:', strong_best[1].name, strong_best[0])


## Evaluate it + Practice

- **Metric.** Track best MSE, solved tasks, and programs searched, and compare it with choosing the smallest grammar program that fits the first example.
- **Cheap sanity check.** D1 must find x^2+1 with MSE zero and near misses 1.0, 1.0, 5.8.
- **Ablation.** remove x^2+c from the grammar and watch the true program become unreachable.
- **Failure signals.** zero training MSE but poor held-out MSE.

### Practice prompts


- Add an absolute-value operator to the grammar.

- Design examples that distinguish x+1 from x^2+1 immediately.

- Cap search by size and compare recovered programs.